In [ ]:
import subprocess
import sys

def _install(packages: list[str]) -> None:
    for pkg in packages:
        print(f"[SETUP] Installing {pkg}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-U", pkg],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.STDOUT,
        )
    print("[SETUP] All packages installed.")

_install(["ultralytics", "ipywidgets", "supervision", "inference", "onnx"])

In [ ]:
import sys
import torch

print(f"[DEBUG] Python: {sys.version}")
print(f"[DEBUG] PyTorch: {torch.__version__}")
print(f"[DEBUG] CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    GPU_COUNT: int = torch.cuda.device_count()
    for i in range(GPU_COUNT):
        name: str = torch.cuda.get_device_name(i)
        props = torch.cuda.get_device_properties(i)
        if hasattr(props, "total_memory"):
            mem_gb: float = float(props.total_memory) / 1e9
        elif hasattr(props, "total_mem"):
            mem_gb = float(props.total_mem) / 1e9
        else:
            raise AttributeError(f"Cannot read GPU memory from {type(props)}")
        print(f"[DEBUG] GPU {i}: {name} ({mem_gb:.1f} GB)")
    print(f"[DEBUG] Total GPUs: {GPU_COUNT}")
else:
    print("[DEBUG] No CUDA GPU detected — running on CPU")

In [ ]:
# Load model
from ultralytics import YOLO

MODEL_PATH: str = "models/weights/best.pt"
model = YOLO(MODEL_PATH)

print(f"[DEBUG] Model loaded: {MODEL_PATH}")
print(f"[DEBUG] Task: {model.task}")
print(f"[DEBUG] Classes ({len(model.names)}): {model.names}")


In [ ]:
# Evaluation on val set
DATA_YAML: str = "data.yaml"
IMAGE_SIZE: int = 640

print("[DEBUG] Running validation on val split...")
val_metrics = model.val(
    data=DATA_YAML,
    split="val",
    imgsz=IMAGE_SIZE,
    batch=16,
    conf=0.001,
    iou=0.6,
    plots=True,
    save_json=True,
)

print(f"[DEBUG] === Val Set Metrics ===")
print(f"  mAP50:     {val_metrics.box.map50:.4f}")
print(f"  mAP50-95:  {val_metrics.box.map:.4f}")
print(f"  Precision:  {val_metrics.box.mp:.4f}")
print(f"  Recall:     {val_metrics.box.mr:.4f}")

print(f"\n[DEBUG] Per-class mAP50:")
for i, class_name in model.names.items():
    ap50: float = val_metrics.box.ap50[i] if i < len(val_metrics.box.ap50) else 0.0
    print(f"  {class_name}: {ap50:.4f}")

In [ ]:
# Evaluation on test set (unseen data)
print("[DEBUG] Running validation on test split...")
test_metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=IMAGE_SIZE,
    batch=16,
    conf=0.001,
    iou=0.6,
    plots=True,
    save_json=True,
)

print(f"[DEBUG] === Test Set Metrics ===")
print(f"  mAP50:     {test_metrics.box.map50:.4f}")
print(f"  mAP50-95:  {test_metrics.box.map:.4f}")
print(f"  Precision:  {test_metrics.box.mp:.4f}")
print(f"  Recall:     {test_metrics.box.mr:.4f}")

print(f"\n[DEBUG] Per-class mAP50:")
for i, class_name in model.names.items():
    ap50: float = test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0.0
    print(f"  {class_name}: {ap50:.4f}")

In [ ]:
# Multi-resolution accuracy comparison (320 vs 416 vs 640)
# Jetson Nano (4GB, 128 CUDA cores, Maxwell) — imgsz=640 is too slow.
# Find the best speed/accuracy trade-off before deploying.

RESOLUTIONS: list[int] = [320, 416, 640]

print("[DEBUG] === Multi-Resolution Accuracy Comparison ===")
print(f"[DEBUG] Testing on test split with resolutions: {RESOLUTIONS}\n")

resolution_results: dict[int, dict[str, float]] = {}

for imgsz in RESOLUTIONS:
    print(f"[DEBUG] Running val at imgsz={imgsz}...")
    metrics = model.val(
        data=DATA_YAML,
        split="test",
        imgsz=imgsz,
        batch=16,
        conf=0.001,
        iou=0.6,
        plots=False,
        verbose=False,
    )
    resolution_results[imgsz] = {
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
    }
    print(f"  mAP50={metrics.box.map50:.4f}  mAP50-95={metrics.box.map:.4f}  P={metrics.box.mp:.4f}  R={metrics.box.mr:.4f}")

print("\n[DEBUG] === Summary Table ===")
print(f"{'imgsz':>6} | {'mAP50':>8} | {'mAP50-95':>10} | {'Precision':>10} | {'Recall':>8} | {'mAP50 loss vs 640':>18}")
print("-" * 80)
baseline_map50: float = resolution_results[640]["mAP50"]
for imgsz in RESOLUTIONS:
    r = resolution_results[imgsz]
    loss_pct: float = ((baseline_map50 - r["mAP50"]) / baseline_map50) * 100 if baseline_map50 > 0 else 0.0
    print(f"{imgsz:>6} | {r['mAP50']:>8.4f} | {r['mAP50-95']:>10.4f} | {r['precision']:>10.4f} | {r['recall']:>8.4f} | {loss_pct:>+17.2f}%")

In [ ]:
# ONNX export for Jetson Nano + DeepStream compatibility
# Requirements:
#   - opset=12 (DeepStream 5.0/6.0 on JetPack 4 requires opset <= 12)
#   - NO dynamic axes (fixed input shape for TensorRT engine build on Nano)
#   - imgsz chosen from multi-resolution test above (pick best trade-off)
#
# After export, transfer .onnx to Nano and rebuild TensorRT engine THERE
# (Maxwell architecture — engine must be built on the target GPU).

NANO_IMGSZ: int = 416  # Change based on multi-resolution results above

print(f"[DEBUG] Exporting ONNX for Jetson Nano — imgsz={NANO_IMGSZ}, opset=12, static shape")
nano_onnx_path: str = model.export(
    format="onnx",
    imgsz=NANO_IMGSZ,
    opset=12,
    dynamic=False,
    simplify=True,
)
print(f"[DEBUG] Nano-compatible ONNX exported: {nano_onnx_path}")

import os
size_mb: float = os.path.getsize(nano_onnx_path) / (1024 * 1024)
print(f"[DEBUG] File size: {size_mb:.2f} MB")
print(f"[DEBUG] Next: SCP this file to Nano, then run trtexec to build TensorRT engine on device")

In [ ]:
# Export model — ONNX (standard deployment format)
print("[DEBUG] Exporting to ONNX...")
onnx_path: str = model.export(format="onnx", imgsz=IMAGE_SIZE)
print(f"[DEBUG] ONNX exported: {onnx_path}")

In [ ]:
# Export model — ONNX FP16 (half precision, smaller + faster)
print("[DEBUG] Exporting to ONNX FP16...")
onnx_fp16_path: str = model.export(format="onnx", imgsz=IMAGE_SIZE, half=True)
print(f"[DEBUG] ONNX FP16 exported: {onnx_fp16_path}")

In [ ]:
# Export model — TensorRT (best performance on NVIDIA GPUs)
print("[DEBUG] Exporting to TensorRT...")
engine_path: str = model.export(format="engine", imgsz=IMAGE_SIZE, half=True)
print(f"[DEBUG] TensorRT exported: {engine_path}")

In [ ]:
# Verify exported model — run inference with TensorRT engine
import time

trt_model = YOLO(engine_path)

print("[DEBUG] Warm-up run...")
trt_model.predict(source="test/images", imgsz=IMAGE_SIZE, conf=0.25, verbose=False, max_det=1)

print("[DEBUG] Benchmarking TensorRT inference on test images...")
start: float = time.perf_counter()
trt_results = trt_model.predict(
    source="test/images",
    imgsz=IMAGE_SIZE,
    conf=0.25,
    verbose=False,
)
elapsed_ms: float = (time.perf_counter() - start) * 1000

total_dets: int = sum(len(r.boxes) for r in trt_results)
print(f"[DEBUG] TensorRT: {len(trt_results)} images, {total_dets} detections, {elapsed_ms:.0f}ms total")
print(f"[DEBUG] Average: {elapsed_ms / len(trt_results):.1f}ms per image")

In [ ]:
# Compare export sizes
import os

exports: dict[str, str] = {
    "ONNX FP32": onnx_path,
    "ONNX FP16": onnx_fp16_path,
    "TensorRT FP16": engine_path,
    "PyTorch (original)": MODEL_PATH,
}

print("[DEBUG] === Export Size Comparison ===")
for name, path in exports.items():
    if os.path.exists(path):
        size_mb: float = os.path.getsize(path) / (1024 * 1024)
        print(f"  {name}: {size_mb:.1f} MB  ({path})")
    else:
        print(f"  {name}: not found ({path})")

In [ ]:
# Install SAHI for tiled inference
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "sahi"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
print("[SETUP] sahi installed.")

In [ ]:
# SAHI — Instantiate detection model with best.pt
from sahi import AutoDetectionModel

SAHI_DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=0.3,
    device=SAHI_DEVICE,
)
print(f"[DEBUG] SAHI detection model loaded from {MODEL_PATH} on {SAHI_DEVICE}")

In [ ]:
# SAHI — Standard prediction (baseline, no slicing)
import os
from sahi.predict import get_prediction

test_images: list[str] = [
    os.path.join("test/images", f)
    for f in os.listdir("test/images")
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
][:5]

print(f"[DEBUG] Running standard prediction on {len(test_images)} test images...")
for img_path in test_images:
    result = get_prediction(img_path, detection_model)
    print(f"[DEBUG]   {os.path.basename(img_path)}: {len(result.object_prediction_list)} detections")

In [ ]:
# SAHI — Sliced prediction (tiled inference for small objects)
from sahi.predict import get_sliced_prediction

SLICE_HEIGHT: int = 256
SLICE_WIDTH: int = 256
OVERLAP_HEIGHT_RATIO: float = 0.2
OVERLAP_WIDTH_RATIO: float = 0.2

print(f"[DEBUG] Running sliced prediction — {SLICE_HEIGHT}x{SLICE_WIDTH}, overlap={OVERLAP_HEIGHT_RATIO}")
for img_path in test_images:
    sliced_result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=SLICE_HEIGHT,
        slice_width=SLICE_WIDTH,
        overlap_height_ratio=OVERLAP_HEIGHT_RATIO,
        overlap_width_ratio=OVERLAP_WIDTH_RATIO,
    )
    print(f"[DEBUG]   {os.path.basename(img_path)}: {len(sliced_result.object_prediction_list)} detections")

In [ ]:
# SAHI — Visualize sliced result on last image
sliced_result.export_visuals(export_dir="visualizations/")
print("[DEBUG] Sliced visualization exported to visualizations/prediction_visual.png")

try:
    from IPython.display import Image, display
    display(Image("visualizations/prediction_visual.png"))
except ImportError:
    print("[DEBUG] Open visualizations/prediction_visual.png manually")

In [ ]:
# SAHI — Export results to COCO format
coco_annotations = sliced_result.to_coco_annotations()[:5]
print(f"[DEBUG] COCO annotations (first 5): {coco_annotations}")

coco_predictions = sliced_result.to_coco_predictions(image_id=1)[:5]
print(f"[DEBUG] COCO predictions (first 5): {coco_predictions}")

In [ ]:
# SAHI — Batch sliced prediction on entire test directory
from sahi.predict import predict

predict(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    model_device=SAHI_DEVICE,
    model_confidence_threshold=0.4,
    source="test/images",
    slice_height=SLICE_HEIGHT,
    slice_width=SLICE_WIDTH,
    overlap_height_ratio=OVERLAP_HEIGHT_RATIO,
    overlap_width_ratio=OVERLAP_WIDTH_RATIO,
)
print("[DEBUG] Batch sliced prediction complete on test/images")